# 第18講：学習データ・テストデータ・評価指標

機械学習の目的は、手元の過去データを丸暗記することではなく、**まだ答えが分からない新しいデータ**を正確に予測すること（**汎化性能**）にある。
このノートブックでは、第16講と同じ高校の擬似データ（`mhigh.csv`・677人）を使いながら、次の4つを順番に体験する。

1. **なぜデータを分けるのか**：無作為（ランダム）に学習データ（80%）とテストデータ（20%）に分ける
2. **数値を当てる予測の評価指標**：予測値と誤差、**MAE**（平均絶対誤差）と **RMSE**（二乗平均平方根誤差）
3. **過学習と汎化性能**：練習問題を丸暗記して本番で失敗する現象（過学習）と、交差検証
4. **〇×を当てる予測の評価指標**：**混同行列**（TP・FP・FN・TN）、**正答率**・**適合率**・**再現率**


## 0. 準備

最初に1回だけ実行する。グラフの軸に日本語を出すため、Noto Sans JP を登録する。


In [ ]:
import piplite
import pyodide
await piplite.install("seaborn")

import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
%matplotlib inline

from pyodide.http import pyfetch
resp = await pyfetch(
    "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/NotoSansJP%5Bwght%5D.ttf"
)
with open("NotoSansJP.ttf", "wb") as f:
    f.write(await resp.bytes())
fm.fontManager.addfont("NotoSansJP.ttf")

sns.set_theme(style="darkgrid", font="Noto Sans JP")


## 1. データの読み込みと無作為な分割

`mhigh.csv`（生徒677人）を読み込み、列名を英語に揃える。
この表は最初から**成績合計（`total`）が高い順**に並んでいることに注意する。


In [ ]:
url = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/mhigh.csv"
data = pd.read_csv(pyodide.http.open_url(url))
data.columns = ["name", "katakana", "sex", "grade",
                "japanese", "math", "total", "height", "weight", "class"]
data.head()


手元のデータを、練習用の**学習データ**（80%：542人）と本番試験用の**テストデータ**（20%：135人）に分ける。
`sample(frac=0.8, random_state=42)` で無作為に80%を選び、`drop(train.index)` で残りの20%を取り出す。


In [ ]:
train = data.sample(frac=0.8, random_state=42)
test = data.drop(train.index)
print("学習データの人数:", len(train), "テストデータの人数:", len(test))


無作為に分けた場合と、上から順番（`iloc`）に切った場合で、数学（`math`）の平均点を比べてみよう。


In [ ]:
print("【無作為に分割】学習データの数学平均:", train["math"].mean().round(1),
      "テストデータの数学平均:", test["math"].mean().round(1))

bad_train = data.iloc[:542]
bad_test = data.iloc[542:]
print("【上から順番に分割】先頭542人の数学平均:", bad_train["math"].mean().round(1),
      "残り135人の数学平均:", bad_test["math"].mean().round(1))


無作為に分ければ学習データ **56.4点**・テストデータ **56.3点** と同じ性質で揃うが、上から順番に切ると **61.0点** と **37.9点** に大きく偏ってしまう。
データを分けるときは必ず無作為にシャッフルしてから分ける。


## 2. 数値を当てる予測の評価指標（MAE と RMSE）

まずスライドの4人の小テストの例で、**MAE**（平均絶対誤差）と **RMSE**（二乗平均平方根誤差）の違いを確認しよう。
- **モデルA**：4人とも4点ずつ外れた（誤差 `+4, -4, +4, -4`）
- **モデルB**：3人は誤差0点だが、1人だけ16点の大ハズレを出した（誤差 `0, 0, 0, -16`）


In [ ]:
err_a = pd.Series([4, -4, 4, -4])
err_b = pd.Series([0, 0, 0, -16])

print("モデルA（そのまま平均）:", err_a.mean())
print("モデルA → MAE:", err_a.abs().mean(), "RMSE:", (err_a ** 2).mean() ** 0.5)
print("モデルB → MAE:", err_b.abs().mean(), "RMSE:", (err_b ** 2).mean() ** 0.5)


誤差をそのまま平均するとプラスとマイナスが打ち消しあって `0.0` になってしまう。
また、モデルBのように「1人だけ大ハズレ」があると、2乗する **RMSE** が `4.0` から `8.0` に跳ね上がって大失敗を警告してくれる。

次に、高校データのテストデータ（135人）に対して、国語の点数から数学の点数を予測する直線式 `31 + 0.5 * test["japanese"]` の **MAE** と **RMSE** を計算する。


In [ ]:
pred = 31 + 0.5 * test["japanese"]
err = test["math"] - pred
mae = err.abs().mean()
rmse = (err ** 2).mean() ** 0.5
print("テストデータの MAE:", round(mae, 2), "点")
print("テストデータの RMSE:", round(rmse, 2), "点")


## 3. 過学習（丸暗記の失敗）と交差検証を体験する

`scikit-learn` を使って、国語・身長・体重の3列（`["japanese", "height", "weight"]`）から数学の点数を予測する2つのモデルを比べる。
- `LinearRegression`：全体の傾向をシンプルな直線で表すモデル
- `DecisionTreeRegressor`（深さ制限なし）：学習データの542人を徹底的に細かく枝分けして**丸暗記**するモデル


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

cols = ["japanese", "height", "weight"]
m_lin = LinearRegression().fit(train[cols], train["math"])
m_tree = DecisionTreeRegressor(random_state=42).fit(train[cols], train["math"])

print("直線モデル（学習データ MAE）:", round(mean_absolute_error(train["math"], m_lin.predict(train[cols])), 2))
print("直線モデル（テストデータ MAE）:", round(mean_absolute_error(test["math"], m_lin.predict(test[cols])), 2))
print("丸暗記の決定木（学習データ MAE）:", round(mean_absolute_error(train["math"], m_tree.predict(train[cols])), 2))
print("丸暗記の決定木（テストデータ MAE）:", round(mean_absolute_error(test["math"], m_tree.predict(test[cols])), 2))


丸暗記した決定木は、学習データ（練習問題）では誤差 **0.0点** なのに、テストデータ（本番試験）では誤差が **14.41点** に悪化している。これが**過学習**（overfitting）である。

さらに、1回だけの分割ではなく、シャッフル済みの `train` を5等分してテスト役を順番に入れ替える **5分割交差検証**（cross validation）でも比べてみよう。


In [ ]:
from sklearn.model_selection import cross_val_score

scores_lin = -cross_val_score(LinearRegression(), train[cols], train["math"],
                              cv=5, scoring="neg_mean_absolute_error")
scores_tree = -cross_val_score(DecisionTreeRegressor(random_state=42), train[cols], train["math"],
                               cv=5, scoring="neg_mean_absolute_error")
print("直線モデルの5回平均 MAE:", round(scores_lin.mean(), 2))
print("丸暗記決定木の5回平均 MAE:", round(scores_tree.mean(), 2))


## 4. 〇×を当てる予測の評価指標（混同行列・正答率・適合率・再現率）

テストデータ（135人）において、「数学が60点以上なら `1`（正例）、60点未満なら `0`（負例）」を実際の答え `actual` とする。
これに対して、「国語が55点以上なら数学も60点以上（`1`）と予測する」ルール `pred` を作り、4通りの人数（**TP・FP・FN・TN**）を数える。


In [ ]:
actual = (test["math"] >= 60).astype(int)
pred = (test["japanese"] >= 55).astype(int)

tp = ((actual == 1) & (pred == 1)).sum()
fp = ((actual == 0) & (pred == 1)).sum()
fn = ((actual == 1) & (pred == 0)).sum()
tn = ((actual == 0) & (pred == 0)).sum()

print(f"TP（正しく1を発見）: {tp}人")
print(f"FP（空振り・誤報） : {fp}人")
print(f"FN（見逃し）       : {fn}人")
print(f"TN（正しく0を見極め）: {tn}人")


この4つの人数から、**正答率（Accuracy）**・**適合率（Precision：空振りの少なさ）**・**再現率（Recall：見逃しの少なさ）** を計算する。


In [ ]:
accuracy = (tp + tn) / len(test)
precision = tp / (tp + fp)
recall = tp / (tp + fn)

print("正答率（Accuracy） :", round(accuracy, 3))
print("適合率（Precision）:", round(precision, 3))
print("再現率（Recall）   :", round(recall, 3))


## 実習①：学習データとテストデータへの分割

次の4つを順番に実行する。

1. `mhigh.csv` を読み込み、列名を `name`, `katakana`, `sex`, `grade`, `japanese`, `math`, `total`, `height`, `weight`, `class` に揃える
2. `sample(frac=0.8, random_state=42)` と `drop` を使い、学習データ `train`（542人）とテストデータ `test`（135人）に分ける
3. `train` と `test` の国語（`japanese`）の平均点をそれぞれ求め、ほぼ同じ値（約50点）になっていることを確認する
4. 上から順番に切った `data.iloc[:542]` と `data.iloc[542:]` の国語の平均点を求め、どれだけ偏るか確認する


## 実習②：数値予測の評価指標（MAE・RMSE）と過学習

実習①で作った `train` と `test` を使い、次の4つを実行する。

1. テストデータ全員の数学の点数を「学習データの数学平均点（`train["math"].mean()`）」と予測したときの、テストデータの **MAE** と **RMSE** を求める
2. 直線式 `31 + 0.5 * test["japanese"]` で予測したときの、テストデータの **MAE** と **RMSE** を求め、1より誤差が小さくなることを確認する
3. `DecisionTreeRegressor(random_state=42)` に `["japanese", "height", "weight"]` を学習させ、学習データとテストデータの **MAE** を比較する
4. `DecisionTreeRegressor(max_depth=3, random_state=42)` で深さを3に制限し、テストデータの **MAE** がどう改善するか確認する


## 実習③：分類の評価指標（混同行列・正答率・適合率・再現率）

実習①で作った `test` を使い、次の4つを実行する。

1. テストデータ `test` で「数学が60点以上なら `1`」を正解 `actual` とし、「国語が **60点以上** なら `1`」と予測したときの **TP・FP・FN・TN** の人数を数える
2. 上の予測の **正答率（Accuracy）・適合率（Precision）・再現率（Recall）** を小数第3位まで求める
3. 予測の基準を「国語が **50点以上** なら `1`」に緩めたときの **適合率** と **再現率** を求める
4. 基準を60点から50点に緩めると、適合率と再現率がそれぞれどう変化したか1〜2文でまとめる
